# 03 m9_hybrid Development

This exploratory notebook develops the `m9_hybrid` candidate-window model described in `2026-06-24_m9_PRD_v1.md`.

**Purpose.** Test whether an m7-style deterministic candidate generator plus a lean XGBoost candidate scorer can improve real-site reverse power flow (RPF) detection. The notebook is isolated from the main journal workflow.

**Inputs.** Final Alpha and Beta parquet datasets from `publication/2_journal_article/dataset/final/`, plus the journal v2 config for column names and m7 parameters.

**Outputs.** Local misc-only artifacts under `notebooks/99_Misc/outputs/03_m9_hybrid_development/`. These outputs are ignored by git.

**Default mode.** `RUN_FULL_M9 = False`, so the notebook runs a fast smoke workflow. Flip this flag only when ready to run the complete Alpha LOSO and Beta transfer experiment.

## 1. Imports, Controls, And Paths

This section keeps the workflow isolated. It resolves the repo root dynamically, loads final datasets directly from the journal config, and creates local misc output folders. The control flags below are the only intended switches for the first exploratory implementation.

In [ ]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import matplotlib
import numpy as np
import pandas as pd
import yaml

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from xgboost import XGBClassifier

RUN_FULL_M9 = False
RANDOM_SEED = 9
SMOKE_POSITIVE_DAYS = 180
SMOKE_NEGATIVE_DAYS = 180
MAX_NEGATIVES_PER_DAY = 10
BOUNDARY_TOLERANCE_MINUTES = 30
THRESHOLD_GRID = np.round(np.linspace(0.05, 0.95, 19), 3)

# m9 v1 deliberately uses m7 candidates only: no smoothing, no fallback, no cap.
SEARCH_START_HOUR = 6
SEARCH_END_HOUR = 18
MIN_DURATION_MINUTES = 30
MAX_DURATION_MINUTES = 8 * 60
EPS = 1e-9

PALETTE = {
    "orange": "#eb932c",
    "dark_blue": "#22303d",
    "grey": "#2F4D67",
    "light_grey": "#5C7D99",
    "light_white": "#ebe3e3",
}
plt.rcParams.update({
    "font.family": "Arial",
    "axes.edgecolor": PALETTE["dark_blue"],
    "axes.labelcolor": PALETTE["dark_blue"],
    "axes.titlecolor": PALETTE["dark_blue"],
    "xtick.color": PALETTE["dark_blue"],
    "ytick.color": PALETTE["dark_blue"],
})


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "publication" / "2_journal_article" / "config" / "experiment_config.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find PyNRPF repo root from current working directory.")


REPO_ROOT = find_repo_root()
ARTICLE_ROOT = REPO_ROOT / "publication" / "2_journal_article"
MISC_DIR = ARTICLE_ROOT / "notebooks" / "99_Misc"
OUTPUT_ROOT = MISC_DIR / "outputs" / "03_m9_hybrid_development"
INTERMEDIATE_DIR = OUTPUT_ROOT / "intermediate"
METRICS_DIR = OUTPUT_ROOT / "metrics"
FIGURES_DIR = OUTPUT_ROOT / "figures"
HTML_DIR = OUTPUT_ROOT / "html"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [INTERMEDIATE_DIR, METRICS_DIR, FIGURES_DIR, HTML_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = ARTICLE_ROOT / "config" / "experiment_config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as fh:
    CFG = yaml.safe_load(fh)

print("Repo root:", REPO_ROOT)
print("Article root:", ARTICLE_ROOT)
print("Output root:", OUTPUT_ROOT)
print("RUN_FULL_M9:", RUN_FULL_M9)


## 2. Dataset Loading And Utility Helpers

These helpers intentionally duplicate only the small amount of loading and scoring logic needed for misc experimentation. Nothing here modifies the production package or the main journal helper module.

In [ ]:
EXPECTED_COLUMNS = [
    "substation_id",
    "date",
    "timestamp",
    "net_load_MW",
    "solar_MW",
    "label_interval",
    "label_day",
]

COUNT_COLUMNS = ["support", "positive_support", "tp", "fp", "fn", "tn"]
SCORE_COLUMNS = ["precision", "recall", "f1"]


def write_csv(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def load_final_dataset(dataset_key: str) -> pd.DataFrame:
    rel = CFG["paths"][f"{dataset_key}_dataset_path"]
    path = ARTICLE_ROOT / rel
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    missing = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{dataset_key} dataset missing columns: {missing}")
    df = df[EXPECTED_COLUMNS].copy()
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    if getattr(ts.dt, "tz", None) is not None:
        ts = ts.dt.tz_localize(None)
    df["timestamp"] = ts
    df["date"] = df["date"].astype(str)
    df["label_interval"] = df["label_interval"].astype(bool)
    df["label_day"] = df.groupby(["substation_id", "date"])["label_interval"].transform("any")
    df = df.sort_values(["substation_id", "timestamp"]).reset_index(drop=True)
    if df["timestamp"].isna().any():
        raise ValueError(f"{dataset_key} dataset contains unparsable timestamps.")
    return df


def binary_metrics(y_true: Iterable[Any], y_pred: Iterable[Any]) -> dict[str, Any]:
    true = np.asarray(list(y_true), dtype=bool)
    pred = np.asarray(list(y_pred), dtype=bool)
    tp = int((true & pred).sum())
    fp = int((~true & pred).sum())
    fn = int((true & ~pred).sum())
    tn = int((~true & ~pred).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "support": int(len(true)),
        "positive_support": int(true.sum()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def metric_rows_from_decoded(decoded_days: pd.DataFrame, interval_frame: pd.DataFrame, dataset: str, fold_id: str) -> pd.DataFrame:
    rows = []
    rows.append({"dataset": dataset, "fold_id": fold_id, "level": "day", **binary_metrics(decoded_days["label_day"], decoded_days["pred_day"])})
    daytime = interval_frame["hour"].between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both")
    rows.append({
        "dataset": dataset,
        "fold_id": fold_id,
        "level": "interval",
        **binary_metrics(interval_frame.loc[daytime, "label_interval"], interval_frame.loc[daytime, "pred_interval"]),
    })
    return pd.DataFrame(rows)


def true_windows(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (site, date), grp in df.groupby(["substation_id", "date"], sort=False):
        labelled = grp.loc[grp["label_interval"]]
        rows.append({
            "substation_id": site,
            "date": date,
            "label_day": not labelled.empty,
            "true_start": labelled["timestamp"].iloc[0] if not labelled.empty else pd.NaT,
            "true_end": labelled["timestamp"].iloc[-1] if not labelled.empty else pd.NaT,
        })
    return pd.DataFrame(rows)


def alpha_site_order(alpha: pd.DataFrame) -> list[str]:
    summary = (
        alpha.groupby("substation_id", as_index=False)
        .agg(rpf_days=("label_day", "sum"), rpf_intervals=("label_interval", "sum"))
        .sort_values(["rpf_days", "rpf_intervals", "substation_id"], ascending=[False, False, True])
    )
    return summary["substation_id"].tolist()


def filter_site_days(df: pd.DataFrame, site_days: pd.DataFrame) -> pd.DataFrame:
    keys = site_days[["substation_id", "date"]].drop_duplicates()
    return df.merge(keys, on=["substation_id", "date"], how="inner")


def select_smoke_days(alpha: pd.DataFrame, n_pos: int = SMOKE_POSITIVE_DAYS, n_neg: int = SMOKE_NEGATIVE_DAYS) -> pd.DataFrame:
    day_df = alpha[["substation_id", "date", "label_day"]].drop_duplicates()
    ordered_sites = alpha_site_order(alpha)
    day_df["site_rank"] = day_df["substation_id"].map({site: i for i, site in enumerate(ordered_sites)})
    day_df = day_df.sort_values(["site_rank", "date"])
    selected = pd.concat([day_df.loc[day_df["label_day"]].head(n_pos), day_df.loc[~day_df["label_day"]].head(n_neg)], ignore_index=True)
    return selected[["substation_id", "date", "label_day"]]


def timestamp_to_hhmm(value: pd.Timestamp | pd.NaT) -> str:
    if pd.isna(value):
        return ""
    return pd.Timestamp(value).strftime("%H:%M")


## 3. m7-Style Candidate Generation

`run_m7()` currently returns only the selected relaxed interval. For m9 we need the candidate list before that final selection. The function below mirrors the m7 candidate-peak logic, but emits one candidate window per local net-load peak. It uses raw values, does not smooth, does not create fallback windows, and does not cap the candidate count.

In [ ]:
NS_PER_HOUR = 3_600_000_000_000
NS_PER_MIN = 60_000_000_000
NS_PER_SEC = 1_000_000_000
DAY_START_NS = SEARCH_START_HOUR * NS_PER_HOUR
DAY_END_NS = SEARCH_END_HOUR * NS_PER_HOUR


def parse_hhmm(value: str) -> tuple[int, int]:
    parts = str(value).strip().split(":")
    return int(parts[0]), int(parts[1])


def pick_max(vals: np.ndarray, ts_i64: np.ndarray, ref_i64: int) -> int | None:
    if len(vals) == 0:
        return None
    dist = np.abs(ts_i64 - ref_i64)
    order = np.lexsort((ts_i64, dist, -vals))
    return int(order[0])


def pair_is_daytime(left_ts: int, right_ts: int, midnight: int) -> bool:
    left_delta = left_ts - midnight
    right_delta = right_ts - midnight
    return DAY_START_NS <= left_delta <= DAY_END_NS and DAY_START_NS <= right_delta <= DAY_END_NS


def ns_to_timestamp(value: int) -> pd.Timestamp:
    return pd.Timestamp(np.datetime64(value, "ns"))


def generate_m7_peak_candidates(df: pd.DataFrame, cfg: dict[str, Any]) -> pd.DataFrame:
    m7 = cfg["correction"]["m7_threshold"]
    tb_h, tb_m = parse_hhmm(m7["solar_peak_tiebreak_time"])
    tb_offset = tb_h * NS_PER_HOUR + tb_m * NS_PER_MIN
    win_ns = int(m7["peak_window_minutes"]) * 60 * NS_PER_SEC
    rows = []

    for (site, date), grp in df.groupby(["substation_id", "date"], sort=False):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        ts_g = grp["timestamp"].values.astype("datetime64[ns]").astype(np.int64)
        mw_g = grp["net_load_MW"].to_numpy(dtype=float)
        solar_g = grp["solar_MW"].to_numpy(dtype=float)
        midnight = int(np.datetime64(str(date), "ns"))

        skip_reason = None
        if np.any(np.isnan(mw_g)):
            skip_reason = "missing_net_load"
        elif np.any(mw_g < 0):
            skip_reason = "negative_net_load"
        secs = (ts_g - midnight) / NS_PER_SEC
        midday = (secs >= SEARCH_START_HOUR * 3600) & (secs < SEARCH_END_HOUR * 3600)
        if skip_reason is None and midday.sum() < 3:
            skip_reason = "insufficient_midday"
        if skip_reason is not None:
            rows.append({"substation_id": site, "date": date, "candidate_id": -1, "candidate_status": skip_reason})
            continue

        mi = np.where(midday)[0]
        ts_m, mw_m, sol_m = ts_g[mi], mw_g[mi], solar_g[mi]
        sol_ok = ~np.isnan(sol_m)
        if not sol_ok.any() or np.nanmax(sol_m) <= EPS:
            rows.append({"substation_id": site, "date": date, "candidate_id": -1, "candidate_status": "no_solar_peak"})
            continue

        solar_peak_local = pick_max(sol_m[sol_ok], ts_m[sol_ok], midnight + tb_offset)
        solar_peak_ts = int(ts_m[sol_ok][solar_peak_local])
        wl, wh = solar_peak_ts - win_ns, solar_peak_ts + win_ns

        lmax = np.zeros(len(mw_m), dtype=bool)
        if len(mw_m) >= 3:
            lmax[1:-1] = (mw_m[1:-1] > mw_m[:-2]) & (mw_m[1:-1] > mw_m[2:])
        cand_local_indices = np.where((ts_m >= wl) & (ts_m <= wh) & lmax)[0]
        if len(cand_local_indices) == 0:
            rows.append({"substation_id": site, "date": date, "candidate_id": -1, "candidate_status": "no_local_peak", "solar_peak_time": ns_to_timestamp(solar_peak_ts)})
            continue

        candidate_counter = 0
        day_p95_solar = float(np.nanpercentile(solar_g[np.isfinite(solar_g)], 95)) if np.isfinite(solar_g).any() else 0.0
        meaningful_solar_threshold = max(EPS, 0.05 * day_p95_solar)
        for local_idx in cand_local_indices:
            peak_ts = int(ts_m[local_idx])
            peak_mw = float(mw_m[local_idx])
            left_mask = ts_g < peak_ts
            right_mask = ts_g > peak_ts
            if not left_mask.any() or not right_mask.any():
                continue
            left_positions = np.where(left_mask)[0]
            right_positions = np.where(right_mask)[0]
            left_pos = int(left_positions[np.argmin(mw_g[left_positions])])
            right_pos = int(right_positions[np.argmin(mw_g[right_positions])])
            left_ts, right_ts = int(ts_g[left_pos]), int(ts_g[right_pos])
            left_mw, right_mw = float(mw_g[left_pos]), float(mw_g[right_pos])
            if not pair_is_daytime(left_ts, right_ts, midnight):
                continue
            duration_min_between_minima = (right_ts - left_ts) / NS_PER_MIN
            if duration_min_between_minima < MIN_DURATION_MINUTES or duration_min_between_minima > MAX_DURATION_MINUTES:
                continue
            interval_mask = (ts_g > left_ts) & (ts_g < right_ts)
            if not interval_mask.any():
                continue
            interval_positions = np.where(interval_mask)[0]
            solar_inside = solar_g[interval_positions]
            if not np.isfinite(solar_inside).any() or np.nanmax(solar_inside) <= meaningful_solar_threshold:
                continue
            rows.append({
                "substation_id": site,
                "date": date,
                "candidate_id": candidate_counter,
                "candidate_status": "candidate",
                "left_min_time": ns_to_timestamp(left_ts),
                "right_min_time": ns_to_timestamp(right_ts),
                "pred_start": ns_to_timestamp(int(ts_g[interval_positions[0]])),
                "pred_end": ns_to_timestamp(int(ts_g[interval_positions[-1]])),
                "peak_time": ns_to_timestamp(peak_ts),
                "solar_peak_time": ns_to_timestamp(solar_peak_ts),
                "left_min_MW": left_mw,
                "right_min_MW": right_mw,
                "candidate_peak_net_load_MW": peak_mw,
                "bounce_height_MW": peak_mw - ((left_mw + right_mw) / 2.0),
                "duration_minutes": len(interval_positions) * 15,
            })
            candidate_counter += 1
        if candidate_counter == 0:
            rows.append({"substation_id": site, "date": date, "candidate_id": -1, "candidate_status": "no_valid_candidate_after_filters", "solar_peak_time": ns_to_timestamp(solar_peak_ts)})

    out = pd.DataFrame(rows)
    for col in ["left_min_time", "right_min_time", "pred_start", "pred_end", "peak_time", "solar_peak_time"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce")
    return out


## 4. Candidate Features And Labels

This section builds the lean v1 feature set and candidate labels. The diagnostic columns `iou_with_true`, `start_error_minutes`, and `end_error_minutes` are saved but excluded from predictors.

In [ ]:
def finite_percentile(values: np.ndarray, q: float, default: float = 0.0) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return default if len(values) == 0 else float(np.nanpercentile(values, q))


def bell_shape_score(values: np.ndarray, require_positive_peak: bool = False) -> float:
    values = np.asarray(values, dtype=float)
    if len(values) < 4 or not np.isfinite(values).all():
        return 0.0
    peak = float(np.nanmax(values))
    if require_positive_peak and peak <= EPS:
        return 0.0
    peak_idx = int(np.nanargmax(values))
    left_diff = np.diff(values[: peak_idx + 1])
    right_diff = np.diff(values[peak_idx:])
    left_score = float(np.mean(left_diff >= 0)) if len(left_diff) else 0.0
    right_score = float(np.mean(right_diff <= 0)) if len(right_diff) else 0.0
    return 0.5 * left_score + 0.5 * right_score


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 2:
        return 0.0
    a, b = a[mask], b[mask]
    if np.nanstd(a) <= EPS or np.nanstd(b) <= EPS:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def interval_iou(start_a: pd.Timestamp, end_a: pd.Timestamp, start_b: pd.Timestamp, end_b: pd.Timestamp) -> float:
    if pd.isna(start_a) or pd.isna(end_a) or pd.isna(start_b) or pd.isna(end_b):
        return 0.0
    a0, a1 = pd.Timestamp(start_a), pd.Timestamp(end_a) + pd.Timedelta(minutes=15)
    b0, b1 = pd.Timestamp(start_b), pd.Timestamp(end_b) + pd.Timedelta(minutes=15)
    overlap = max(pd.Timedelta(0), min(a1, b1) - max(a0, b0)).total_seconds() / 60
    union = (max(a1, b1) - min(a0, b0)).total_seconds() / 60
    return float(overlap / union) if union > 0 else 0.0


def build_candidate_features(df: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    candidate_rows = candidates.loc[candidates["candidate_status"].eq("candidate")].copy()
    if candidate_rows.empty:
        return candidate_rows
    feature_rows = []
    for (site, date), grp in df.groupby(["substation_id", "date"], sort=False):
        day_candidates = candidate_rows.loc[candidate_rows["substation_id"].eq(site) & candidate_rows["date"].eq(str(date))]
        if day_candidates.empty:
            continue
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        ts = grp["timestamp"]
        net = grp["net_load_MW"].to_numpy(dtype=float)
        solar = grp["solar_MW"].to_numpy(dtype=float)
        daytime = ts.dt.hour.between(SEARCH_START_HOUR, SEARCH_END_HOUR, inclusive="both").to_numpy()
        net_daytime = net[daytime]
        solar_daytime = solar[daytime]
        net_scale = max(abs(finite_percentile(net_daytime, 95)), abs(finite_percentile(net_daytime, 5)), EPS)
        solar_scale = max(finite_percentile(solar_daytime, 95), EPS)
        net_norm = net / net_scale
        daily_solar_peak_time = ts.iloc[int(np.nanargmax(np.where(np.isfinite(solar), solar, -np.inf)))] if np.isfinite(solar).any() else pd.NaT
        missing_net_day = int(np.isnan(net).sum())
        missing_solar_day = int(np.isnan(solar).sum())
        for _, cand in day_candidates.iterrows():
            start = pd.Timestamp(cand["pred_start"])
            end = pd.Timestamp(cand["pred_end"])
            mask = (ts >= start) & (ts <= end)
            idx = np.where(mask.to_numpy())[0]
            if len(idx) == 0:
                continue
            c_net = net[idx]
            c_solar = solar[idx]
            c_net_norm = net_norm[idx]
            ds = np.diff(c_solar)
            dn = np.diff(c_net)
            diff_mask = np.isfinite(ds) & np.isfinite(dn)
            if diff_mask.any():
                same_sign_fraction = float(np.mean((ds[diff_mask] * dn[diff_mask]) > 0))
                mean_derivative_product = float(np.nanmean(ds[diff_mask] * dn[diff_mask]))
                ramp_up = diff_mask & (ds > 0)
                ramp_down = diff_mask & (ds < 0)
                ramp_up_comovement = float(np.mean(dn[ramp_up] > 0)) if ramp_up.any() else 0.0
                ramp_down_comovement = float(np.mean(dn[ramp_down] < 0)) if ramp_down.any() else 0.0
            else:
                same_sign_fraction = mean_derivative_product = ramp_up_comovement = ramp_down_comovement = 0.0
            midpoint = start + (end - start) / 2
            feature_rows.append({
                "substation_id": site,
                "date": str(date),
                "candidate_id": int(cand["candidate_id"]),
                "pred_start": start,
                "pred_end": end,
                "pred_start_hhmm": timestamp_to_hhmm(start),
                "pred_end_hhmm": timestamp_to_hhmm(end),
                "start_hour": start.hour + start.minute / 60.0,
                "end_hour": end.hour + end.minute / 60.0,
                "midpoint_hour": midpoint.hour + midpoint.minute / 60.0,
                "duration_hours": len(idx) * 15 / 60.0,
                "month": start.month,
                "weekday": start.weekday(),
                "weekend_flag": int(start.weekday() >= 5),
                "solar_p95_inside": finite_percentile(c_solar, 95),
                "solar_peak_inside": finite_percentile(c_solar, 100),
                "solar_bell_score": bell_shape_score(c_solar, require_positive_peak=True),
                "net_load_p05_inside": finite_percentile(c_net, 5),
                "net_load_p95_inside": finite_percentile(c_net, 95),
                "net_load_range_inside": finite_percentile(c_net, 100) - finite_percentile(c_net, 0),
                "net_load_peak_positive_inside": max(0.0, finite_percentile(c_net, 100)),
                "net_load_norm_p05_inside": finite_percentile(c_net_norm, 5),
                "net_load_norm_p95_inside": finite_percentile(c_net_norm, 95),
                "net_load_n_shape_score": bell_shape_score(c_net, require_positive_peak=False),
                "solar_net_corr": safe_corr(c_solar, c_net),
                "derivative_same_sign_fraction": same_sign_fraction,
                "mean_derivative_product": mean_derivative_product,
                "ramp_up_comovement": ramp_up_comovement,
                "ramp_down_comovement": ramp_down_comovement,
                "contains_daily_solar_peak": int(pd.notna(daily_solar_peak_time) and start <= daily_solar_peak_time <= end),
                "bounce_height_MW": float(cand.get("bounce_height_MW", np.nan)),
                "distance_midpoint_to_solar_peak_minutes": abs((midpoint - pd.Timestamp(cand["solar_peak_time"])).total_seconds()) / 60 if pd.notna(cand.get("solar_peak_time")) else np.nan,
                "missing_intervals_inside": int(np.isnan(c_net).sum() + np.isnan(c_solar).sum()),
                "missing_net_load_day": missing_net_day,
                "missing_solar_day": missing_solar_day,
            })
    return pd.DataFrame(feature_rows)


def label_candidate_features(features: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    windows = true_windows(df)
    labelled = features.merge(windows, on=["substation_id", "date"], how="left")
    start_errors, end_errors, ious, positives = [], [], [], []
    for _, row in labelled.iterrows():
        if not bool(row.get("label_day", False)) or pd.isna(row.get("true_start")) or pd.isna(row.get("true_end")):
            start_errors.append(np.nan); end_errors.append(np.nan); ious.append(0.0); positives.append(False); continue
        start_error = abs((pd.Timestamp(row["pred_start"]) - pd.Timestamp(row["true_start"])).total_seconds()) / 60
        end_error = abs((pd.Timestamp(row["pred_end"]) - pd.Timestamp(row["true_end"])).total_seconds()) / 60
        start_errors.append(start_error)
        end_errors.append(end_error)
        ious.append(interval_iou(row["pred_start"], row["pred_end"], row["true_start"], row["true_end"]))
        positives.append(start_error <= BOUNDARY_TOLERANCE_MINUTES and end_error <= BOUNDARY_TOLERANCE_MINUTES)
    labelled["start_error_minutes"] = start_errors
    labelled["end_error_minutes"] = end_errors
    labelled["iou_with_true"] = ious
    labelled["is_positive"] = positives
    return labelled


def candidate_day_summary(df: pd.DataFrame, labelled_features: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    days = true_windows(df)
    candidate_counts = candidates.loc[candidates["candidate_status"].eq("candidate")].groupby(["substation_id", "date"], as_index=False).size().rename(columns={"size": "candidate_count"})
    if labelled_features.empty:
        positive_counts = pd.DataFrame(columns=["substation_id", "date", "positive_candidate_count", "best_iou", "best_start_error_minutes", "best_end_error_minutes"])
    else:
        positive_counts = labelled_features.groupby(["substation_id", "date"], as_index=False).agg(
            positive_candidate_count=("is_positive", "sum"),
            best_iou=("iou_with_true", "max"),
            best_start_error_minutes=("start_error_minutes", "min"),
            best_end_error_minutes=("end_error_minutes", "min"),
        )
    out = days.merge(candidate_counts, on=["substation_id", "date"], how="left").merge(positive_counts, on=["substation_id", "date"], how="left")
    out["candidate_count"] = out["candidate_count"].fillna(0).astype(int)
    out["positive_candidate_count"] = out["positive_candidate_count"].fillna(0).astype(int)
    out["best_iou"] = out["best_iou"].fillna(0.0)
    out["has_positive_candidate"] = out["positive_candidate_count"] > 0
    return out


## 5. Model Training, Decoding, And Metrics

The scorer is a binary XGBoost model over candidate windows. Training keeps all positives, samples negatives for balance, and uses class weighting. Inference always scores every original m7 candidate and decodes at most one window per site-day.

In [ ]:
FEATURE_NUMERIC_COLUMNS = [
    "start_hour", "end_hour", "midpoint_hour", "duration_hours", "weekend_flag",
    "solar_p95_inside", "solar_peak_inside", "solar_bell_score",
    "net_load_p05_inside", "net_load_p95_inside", "net_load_range_inside", "net_load_peak_positive_inside",
    "net_load_norm_p05_inside", "net_load_norm_p95_inside", "net_load_n_shape_score",
    "solar_net_corr", "derivative_same_sign_fraction", "mean_derivative_product", "ramp_up_comovement", "ramp_down_comovement",
    "contains_daily_solar_peak", "bounce_height_MW", "distance_midpoint_to_solar_peak_minutes",
    "missing_intervals_inside", "missing_net_load_day", "missing_solar_day",
]
FEATURE_CATEGORICAL_COLUMNS = ["month", "weekday"]


def make_feature_matrix(df: pd.DataFrame, columns: list[str] | None = None) -> tuple[pd.DataFrame, list[str]]:
    if df.empty:
        empty = pd.DataFrame(columns=columns or [])
        return empty, list(empty.columns)
    base = df[FEATURE_NUMERIC_COLUMNS + FEATURE_CATEGORICAL_COLUMNS].copy()
    base = pd.get_dummies(base, columns=FEATURE_CATEGORICAL_COLUMNS, prefix=FEATURE_CATEGORICAL_COLUMNS, dtype=float)
    base = base.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if columns is not None:
        base = base.reindex(columns=columns, fill_value=0.0)
        return base, columns
    return base, base.columns.tolist()


def sample_training_candidates(labelled: pd.DataFrame, seed: int = RANDOM_SEED, max_negatives_per_day: int = MAX_NEGATIVES_PER_DAY) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    parts = []
    for _, grp in labelled.groupby(["substation_id", "date"], sort=False):
        positives = grp.loc[grp["is_positive"]]
        negatives = grp.loc[~grp["is_positive"]].copy()
        if not positives.empty:
            parts.append(positives)
        if negatives.empty:
            continue
        negatives["_hard_rank_iou"] = negatives["iou_with_true"].fillna(0.0)
        negatives["_hard_rank_boundary"] = negatives[["start_error_minutes", "end_error_minutes"]].fillna(10_000).sum(axis=1)
        negatives["_hard_rank_solar"] = negatives["solar_peak_inside"].fillna(0.0)
        negatives["_random"] = rng.random(len(negatives))
        selected = negatives.sort_values(
            ["_hard_rank_iou", "_hard_rank_boundary", "_hard_rank_solar", "_random"],
            ascending=[False, True, False, True],
        ).head(max_negatives_per_day)
        parts.append(selected.drop(columns=["_hard_rank_iou", "_hard_rank_boundary", "_hard_rank_solar", "_random"]))
    return pd.concat(parts, ignore_index=True) if parts else labelled.iloc[0:0].copy()


@dataclass
class M9ModelBundle:
    model: XGBClassifier
    feature_columns: list[str]
    threshold: float


def train_m9_classifier(labelled: pd.DataFrame, seed: int = RANDOM_SEED, smoke: bool = False) -> M9ModelBundle:
    train_df = sample_training_candidates(labelled, seed=seed)
    if train_df.empty or train_df["is_positive"].nunique() < 2:
        raise ValueError("Training candidates must contain at least one positive and one negative candidate. Inspect candidate generator recall before training.")
    X, feature_columns = make_feature_matrix(train_df)
    y = train_df["is_positive"].astype(int).to_numpy()
    pos = max(1, int(y.sum()))
    neg = max(1, int(len(y) - y.sum()))
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_estimators=80 if smoke else 300,
        max_depth=3 if smoke else 4,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=neg / pos,
        random_state=seed,
        n_jobs=0,
    )
    model.fit(X, y)
    return M9ModelBundle(model=model, feature_columns=feature_columns, threshold=0.5)


def score_candidates(bundle: M9ModelBundle, features: pd.DataFrame) -> pd.DataFrame:
    scored = features.copy()
    if scored.empty:
        scored["candidate_probability"] = []
        return scored
    X, _ = make_feature_matrix(scored, bundle.feature_columns)
    scored["candidate_probability"] = bundle.model.predict_proba(X)[:, 1]
    return scored


def decode_site_days(eval_df: pd.DataFrame, scored_candidates: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    days = true_windows(eval_df)
    if scored_candidates.empty:
        best = pd.DataFrame(columns=["substation_id", "date", "candidate_probability", "pred_start", "pred_end", "candidate_id"])
    else:
        best = (
            scored_candidates.sort_values(["substation_id", "date", "candidate_probability"], ascending=[True, True, False])
            .groupby(["substation_id", "date"], as_index=False)
            .head(1)[["substation_id", "date", "candidate_id", "pred_start", "pred_end", "candidate_probability"]]
        )
    decoded = days.merge(best, on=["substation_id", "date"], how="left")
    decoded["candidate_probability"] = decoded["candidate_probability"].fillna(0.0)
    decoded["pred_day"] = decoded["candidate_probability"] >= threshold
    decoded.loc[~decoded["pred_day"], ["pred_start", "pred_end"]] = pd.NaT
    decoded["pred_start_hhmm"] = decoded["pred_start"].map(timestamp_to_hhmm)
    decoded["pred_end_hhmm"] = decoded["pred_end"].map(timestamp_to_hhmm)
    interval_frame = eval_df.copy()
    interval_frame = interval_frame.merge(decoded[["substation_id", "date", "pred_day", "pred_start", "pred_end"]], on=["substation_id", "date"], how="left")
    interval_frame["pred_interval"] = (
        interval_frame["pred_day"].fillna(False)
        & (interval_frame["timestamp"] >= interval_frame["pred_start"])
        & (interval_frame["timestamp"] <= interval_frame["pred_end"])
    )
    interval_frame["hour"] = interval_frame["timestamp"].dt.hour
    return decoded, interval_frame


def threshold_sweep(eval_df: pd.DataFrame, scored_candidates: pd.DataFrame, dataset: str, fold_id: str) -> pd.DataFrame:
    rows = []
    for threshold in THRESHOLD_GRID:
        decoded, interval_frame = decode_site_days(eval_df, scored_candidates, float(threshold))
        day = metric_rows_from_decoded(decoded, interval_frame, dataset=dataset, fold_id=fold_id).loc[lambda x: x["level"].eq("day")].iloc[0]
        rows.append({"threshold": float(threshold), **{col: day[col] for col in COUNT_COLUMNS + SCORE_COLUMNS}})
    return pd.DataFrame(rows)


def choose_threshold(eval_df: pd.DataFrame, scored_candidates: pd.DataFrame) -> float:
    sweep = threshold_sweep(eval_df, scored_candidates, dataset="Alpha", fold_id="threshold_selection")
    best = sweep.sort_values(["f1", "precision", "threshold"], ascending=[False, False, False]).iloc[0]
    return float(best["threshold"])


## 6. Diagnostics And Plotting Helpers

The figures are intentionally simple. The goal is to verify that m7 candidates exist, the candidate generator covers labelled RPF days, and the model can be inspected through examples.

In [ ]:
def write_candidate_count_figure(summary: pd.DataFrame, path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(7.2, 4.0))
    ax.hist(summary["candidate_count"], bins=range(0, int(summary["candidate_count"].max()) + 3), color=PALETTE["orange"], edgecolor="white")
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=PALETTE["light_white"], linewidth=0.8)
    ax.set_title("m9_hybrid m7-candidate count per site-day", fontsize=13)
    ax.set_xlabel("Candidate windows per site-day")
    ax.set_ylabel("Site-days")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return path


def write_candidate_recall_figure(summary: pd.DataFrame, path: Path) -> Path:
    rpf = summary.loc[summary["label_day"]].copy()
    by_site = rpf.groupby("substation_id", as_index=False).agg(rpf_days=("date", "count"), candidate_recall=("has_positive_candidate", "mean"))
    by_site = by_site.sort_values("candidate_recall", ascending=False)
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    ax.bar(by_site["substation_id"], by_site["candidate_recall"] * 100, color=PALETTE["dark_blue"], width=0.7)
    ax.set_ylim(0, 100)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=PALETTE["light_white"], linewidth=0.8)
    ax.set_title("Candidate-generator recall on labelled RPF days", fontsize=13)
    ax.set_xlabel("Site")
    ax.set_ylabel("RPF days with a near-true candidate (%)")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return path


def write_feature_importance_figure(bundle: M9ModelBundle, path: Path, top_n: int = 18) -> Path:
    importance = pd.DataFrame({"feature": bundle.feature_columns, "importance": bundle.model.feature_importances_})
    importance = importance.sort_values("importance", ascending=False).head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7.4, 5.2))
    ax.barh(importance["feature"], importance["importance"], color=PALETTE["grey"])
    ax.set_axisbelow(True)
    ax.grid(axis="x", color=PALETTE["light_white"], linewidth=0.8)
    ax.set_title("m9_hybrid smoke model feature importance", fontsize=13)
    ax.set_xlabel("XGBoost importance")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return path


def write_example_html(eval_df: pd.DataFrame, decoded: pd.DataFrame, candidates: pd.DataFrame, site: str, date: str, path: Path) -> Path:
    day = eval_df.loc[eval_df["substation_id"].eq(site) & eval_df["date"].eq(date)].sort_values("timestamp")
    dec = decoded.loc[decoded["substation_id"].eq(site) & decoded["date"].eq(date)].iloc[0]
    cand_day = candidates.loc[candidates["substation_id"].eq(site) & candidates["date"].eq(date) & candidates["candidate_status"].eq("candidate")]
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["net_load_MW"], mode="lines", name="Raw net load", line=dict(color=PALETTE["dark_blue"])), secondary_y=False)
    fig.add_trace(go.Scatter(x=day["timestamp"], y=day["solar_MW"], mode="lines", name="Solar", line=dict(color=PALETTE["orange"])), secondary_y=True)
    labelled = day.loc[day["label_interval"]]
    if not labelled.empty:
        fig.add_vrect(x0=labelled["timestamp"].iloc[0], x1=labelled["timestamp"].iloc[-1], fillcolor="rgba(235,147,44,0.18)", line_width=0, annotation_text="manual", annotation_position="top left")
    if bool(dec["pred_day"]):
        fig.add_vrect(x0=dec["pred_start"], x1=dec["pred_end"], fillcolor="rgba(47,77,103,0.18)", line_width=0, annotation_text="m9", annotation_position="top right")
    for _, cand in cand_day.iterrows():
        fig.add_vline(x=cand["pred_start"], line_width=1, line_dash="dot", line_color=PALETTE["light_grey"])
        fig.add_vline(x=cand["pred_end"], line_width=1, line_dash="dot", line_color=PALETTE["light_grey"])
    fig.update_layout(title=f"m9_hybrid example | {site} | {date}", template="plotly_white", font=dict(family="Arial", color=PALETTE["dark_blue"]), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0), height=520)
    fig.update_yaxes(title_text="Net load (MW)", secondary_y=False)
    fig.update_yaxes(title_text="Solar (MW)", secondary_y=True)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(path)
    return path


def write_manifest(payload: dict[str, Any]) -> Path:
    path = MANIFEST_DIR / "run_manifest.json"
    with path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, default=str)
    return path


## 7. Smoke Workflow Or Full Workflow

Smoke mode validates the complete plumbing on a deterministic Alpha subset. Full mode runs complete Alpha LOSO and Beta transfer, but it is intentionally disabled by default.

In [ ]:
t0 = time.perf_counter()
alpha = load_final_dataset("alpha")
beta = load_final_dataset("beta")
print(f"Loaded Alpha rows={len(alpha):,}, sites={alpha['substation_id'].nunique()}")
print(f"Loaded Beta rows={len(beta):,}, sites={beta['substation_id'].nunique()}")

if not RUN_FULL_M9:
    selected_days = select_smoke_days(alpha)
    smoke_alpha = filter_site_days(alpha, selected_days)
    print(f"Smoke Alpha rows={len(smoke_alpha):,}, site-days={selected_days.shape[0]:,}")
    candidates = generate_m7_peak_candidates(smoke_alpha, CFG)
    features = build_candidate_features(smoke_alpha, candidates)
    labelled = label_candidate_features(features, smoke_alpha)
    day_summary = candidate_day_summary(smoke_alpha, labelled, candidates)
    if labelled.empty or labelled["is_positive"].sum() == 0:
        raise RuntimeError("Smoke candidate generation produced no positive candidates. Increase smoke sample size or inspect m7 candidate recall.")
    bundle = train_m9_classifier(labelled, smoke=True)
    scored = score_candidates(bundle, features)
    threshold = choose_threshold(smoke_alpha, scored)
    bundle.threshold = threshold
    decoded, interval_frame = decode_site_days(smoke_alpha, scored, threshold)
    metrics = metric_rows_from_decoded(decoded, interval_frame, dataset="Alpha", fold_id="smoke")
    write_csv(candidates, INTERMEDIATE_DIR / "01_smoke_m7_candidates.csv")
    write_csv(labelled, INTERMEDIATE_DIR / "02_smoke_candidate_features_labels.csv")
    write_csv(day_summary, INTERMEDIATE_DIR / "03_smoke_candidate_day_summary.csv")
    write_csv(scored, INTERMEDIATE_DIR / "04_smoke_scored_candidates.csv")
    write_csv(decoded, INTERMEDIATE_DIR / "05_smoke_decoded_days.csv")
    write_csv(metrics, METRICS_DIR / "01_smoke_metrics.csv")
    write_csv(threshold_sweep(smoke_alpha, scored, dataset="Alpha", fold_id="smoke"), METRICS_DIR / "02_smoke_threshold_sweep.csv")
    count_fig = write_candidate_count_figure(day_summary, FIGURES_DIR / "fig01_smoke_candidate_count_distribution.png")
    recall_fig = write_candidate_recall_figure(day_summary, FIGURES_DIR / "fig02_smoke_candidate_recall_by_site.png")
    importance_fig = write_feature_importance_figure(bundle, FIGURES_DIR / "fig03_smoke_feature_importance.png")
    example = decoded.sort_values(["pred_day", "label_day", "candidate_probability"], ascending=[False, False, False]).iloc[0]
    html_path = write_example_html(smoke_alpha, decoded, candidates, str(example["substation_id"]), str(example["date"]), HTML_DIR / "smoke_example.html")
    manifest = write_manifest({
        "mode": "smoke",
        "run_full_m9": RUN_FULL_M9,
        "elapsed_seconds": time.perf_counter() - t0,
        "threshold": threshold,
        "alpha_rows": int(len(alpha)),
        "beta_rows": int(len(beta)),
        "smoke_rows": int(len(smoke_alpha)),
        "candidate_rows": int((candidates["candidate_status"] == "candidate").sum()),
        "positive_candidate_rows": int(labelled["is_positive"].sum()),
        "outputs": {
            "metrics": ["01_smoke_metrics.csv", "02_smoke_threshold_sweep.csv"],
            "figures": [count_fig.name, recall_fig.name, importance_fig.name],
            "html": [html_path.name],
        },
    })
    print("Smoke metrics:")
    display(metrics)
    print("Selected threshold:", threshold)
    print("Manifest:", manifest)
else:
    print("RUN_FULL_M9=True: running complete Alpha LOSO and Beta transfer.")
    alpha_candidates = generate_m7_peak_candidates(alpha, CFG)
    alpha_features = build_candidate_features(alpha, alpha_candidates)
    alpha_labelled = label_candidate_features(alpha_features, alpha)
    alpha_day_summary = candidate_day_summary(alpha, alpha_labelled, alpha_candidates)
    write_csv(alpha_candidates, INTERMEDIATE_DIR / "01_alpha_m7_candidates.csv")
    write_csv(alpha_labelled, INTERMEDIATE_DIR / "02_alpha_candidate_features_labels.csv")
    write_csv(alpha_day_summary, INTERMEDIATE_DIR / "03_alpha_candidate_day_summary.csv")
    loso_scored_parts, loso_metric_parts = [], []
    sites = alpha_site_order(alpha)
    for holdout in sites:
        train_labelled = alpha_labelled.loc[~alpha_labelled["substation_id"].eq(holdout)].copy()
        holdout_df = alpha.loc[alpha["substation_id"].eq(holdout)].copy()
        holdout_features = alpha_labelled.loc[alpha_labelled["substation_id"].eq(holdout)].drop(columns=["is_positive"], errors="ignore")
        model = train_m9_classifier(train_labelled, smoke=False)
        scored_holdout = score_candidates(model, holdout_features)
        scored_holdout["fold_id"] = f"alpha_holdout_{holdout}"
        loso_scored_parts.append(scored_holdout)
    alpha_oof_scored = pd.concat(loso_scored_parts, ignore_index=True) if loso_scored_parts else pd.DataFrame()
    threshold = choose_threshold(alpha, alpha_oof_scored)
    for holdout in sites:
        holdout_df = alpha.loc[alpha["substation_id"].eq(holdout)].copy()
        scored_holdout = alpha_oof_scored.loc[alpha_oof_scored["substation_id"].eq(holdout)].copy()
        decoded, interval_frame = decode_site_days(holdout_df, scored_holdout, threshold)
        loso_metric_parts.append(metric_rows_from_decoded(decoded, interval_frame, dataset="Alpha", fold_id=f"alpha_holdout_{holdout}"))
    alpha_loso_metrics = pd.concat(loso_metric_parts, ignore_index=True)
    write_csv(alpha_oof_scored, INTERMEDIATE_DIR / "04_alpha_loso_scored_candidates.csv")
    write_csv(alpha_loso_metrics, METRICS_DIR / "01_alpha_loso_metrics.csv")
    write_csv(threshold_sweep(alpha, alpha_oof_scored, dataset="Alpha", fold_id="alpha_oof"), METRICS_DIR / "02_alpha_oof_threshold_sweep.csv")
    final_model = train_m9_classifier(alpha_labelled, smoke=False)
    beta_candidates = generate_m7_peak_candidates(beta, CFG)
    beta_features = build_candidate_features(beta, beta_candidates)
    beta_scored = score_candidates(final_model, beta_features)
    beta_decoded, beta_interval_frame = decode_site_days(beta, beta_scored, threshold)
    beta_metrics = metric_rows_from_decoded(beta_decoded, beta_interval_frame, dataset="Beta", fold_id="beta_transfer")
    beta_site_metrics = pd.concat([
        metric_rows_from_decoded(
            beta_decoded.loc[beta_decoded["substation_id"].eq(site)],
            beta_interval_frame.loc[beta_interval_frame["substation_id"].eq(site)],
            dataset="Beta",
            fold_id=f"beta_site_{site}",
        )
        for site in sorted(beta["substation_id"].unique())
    ], ignore_index=True)
    write_csv(beta_candidates, INTERMEDIATE_DIR / "05_beta_m7_candidates.csv")
    write_csv(beta_scored, INTERMEDIATE_DIR / "06_beta_scored_candidates.csv")
    write_csv(beta_decoded, INTERMEDIATE_DIR / "07_beta_decoded_days.csv")
    write_csv(beta_metrics, METRICS_DIR / "03_beta_transfer_metrics.csv")
    write_csv(beta_site_metrics, METRICS_DIR / "04_beta_site_metrics.csv")
    count_fig = write_candidate_count_figure(alpha_day_summary, FIGURES_DIR / "fig01_alpha_candidate_count_distribution.png")
    recall_fig = write_candidate_recall_figure(alpha_day_summary, FIGURES_DIR / "fig02_alpha_candidate_recall_by_site.png")
    importance_fig = write_feature_importance_figure(final_model, FIGURES_DIR / "fig03_feature_importance.png")
    example = beta_decoded.sort_values(["pred_day", "label_day", "candidate_probability"], ascending=[False, False, False]).iloc[0]
    html_path = write_example_html(beta, beta_decoded, beta_candidates, str(example["substation_id"]), str(example["date"]), HTML_DIR / "beta_example.html")
    manifest = write_manifest({
        "mode": "full",
        "run_full_m9": RUN_FULL_M9,
        "elapsed_seconds": time.perf_counter() - t0,
        "threshold": threshold,
        "alpha_candidate_rows": int((alpha_candidates["candidate_status"] == "candidate").sum()),
        "beta_candidate_rows": int((beta_candidates["candidate_status"] == "candidate").sum()),
        "outputs": {"metrics": ["01_alpha_loso_metrics.csv", "03_beta_transfer_metrics.csv", "04_beta_site_metrics.csv"], "figures": [count_fig.name, recall_fig.name, importance_fig.name], "html": [html_path.name]},
    })
    print("Alpha LOSO metrics:")
    display(alpha_loso_metrics)
    print("Beta transfer metrics:")
    display(beta_metrics)
    print("Selected threshold:", threshold)
    print("Manifest:", manifest)


## 8. Output Inventory

This final cell lists the local artifacts written by the notebook. The files are diagnostic-only and should not be used as publication-ready results unless the method is later promoted into the main journal workflow.

In [ ]:
for folder in [INTERMEDIATE_DIR, METRICS_DIR, FIGURES_DIR, HTML_DIR, MANIFEST_DIR]:
    print(f"\n{folder.relative_to(MISC_DIR)}")
    for item in sorted(folder.glob("*")):
        print(" -", item.name)
